# Working With OpenAI APIs

In this session, we will extensively use the OpenAI API endpoints (for generating one-off responses and conducting chats/conversations, respectively). 

In this section, we will:
1. Make API calls to the `chat.completions` endpoints
2. Make API calls to the `responses` endpoints
3. Modify the prompts and make them more nuanced to perform complex tasks
4. Create a very simple 'AI Tutor'
5. Measure the cost of making API calls via tokens and put guardrails in place to monitor and control costs

## `chat.completions` API

`chat.completions` is not the newest API. It is legacy-style and maintained mainly for backward compatibility. It represents a separate mental model (“chat vs completion”), which no longer reflects how modern GenAI systems work.

So,, earlier responses from OpenAI models were fetched as: `chat.completions.create(...)`

OpenAI has now unified chat, reasoning, tool use, and multimodal outputs under the Responses API. Conceptually, chat is now treated as just one interaction pattern, not a separate API.

`chat.completions` was designed only for chat-style text generation, with limited support for tool calling, multimodal inputs and structured outputs. It still works, but it is not the direction forward.

We first need to install the `openai` Python library. You also need to get an OpenAI API key. Create an OpenAI account and [get an API key here](https://platform.openai.com/account/api-keys).

In [ ]:
# install openai
# !pip install openai

To use OpenAI APIs we need to set an API key. You need to store the API keys in a safe `.env` file and **avoid uploading it to any project or version management platform, such as GitHub**. If your API keys are uploaded on a public platform, there is a high chance it might get picked up by a scraper, and you might incur additional costs.

In [3]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print("Check: Key Loaded successfully.")
else:
    print("Check: Keys Missing! Check your `.env` file.")

Check: Key Loaded successfully.


### Getting Responses from `chat.completions`

We can now use any of the OpenAI models. For natural language (text) and code, for any given model (GPT-3.5, 4, or 5), OpenAI provided chat completion capability via the `chat.completions()` endpoint. This API could be used for chat-like multi-turn conversation and takes the entire conversation history as input and returns the next response.



* We use the cost-effective model `gpt-4.1-nano`, which belongs to the GPT-4 family - this still works well for basic tasks
* `max_completion_tokens` refers to the max number of tokens to be generated
* `temperature` is a number between 0 (most certain/deterministic) and 2 (most random), defaults to 1; this controls the likelihood of choosing the output token other than the one with the highest probability

The `chat.completions()` API requires three main roles to be specified in the API:
1. **System**: This is an instruction that sets the behaviour of the assistant, e.g. "You are a helpful math tutor", or "You are a helpful advisor for financial analysts"
2. **User**: This role represents the end user using the chatbot
3. **Assistant**: This is the chatbot, represents the reponses generated by the assistant

An API call looks like as follows.
- We provide an initial system instruction, a user input and an assistant input to  `messages`, which is a list that contains the entire conversation history. 
- In each subsequent API call, we can also pass on the entire conversation history.

Notice that we are now using the gpt-4.1-nano model (it is a cheaper model as compared to the other ones, more on that later).

In [4]:
# Let's first setup the client
from openai import OpenAI

openai = OpenAI(api_key=OPENAI_API_KEY)     # client

In [5]:
# using the Completion API

# define an input prompt
prompt = '''You are a helpful Python teaching assistant. Explain the various list indexing methods in Python. Provide an
exhaustive summary of the methods describing what they do, sample code for each, and guidelines on when to use which method.
'''

message = [{"role": "user", "content": prompt}]


chat_response = openai.chat.completions.create(
    model="gpt-4.1-nano",
    messages=message,
    max_completion_tokens=200,
    temperature=0.5,
    n=1,
    stop=None,
    frequency_penalty=0,
    presence_penalty=0)

chat_response

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [15]:
print(type(chat_response))

<class 'openai.types.chat.chat_completion.ChatCompletion'>


The API returns a dictionary-like object. Also, notice that the API returns the number of `total_tokens` used (prompt tokens + completion tokens).

<br>

The reply we are interested in is the `text` inside `choices`, which seems to be a list. We can access that as follows:

In [ ]:
# retrieve the response text
print(chat_response.choices[0].message.content)

Certainly! List indexing in Python is a fundamental way to access and manipulate elements within a list. Python offers several ways to index and slice lists, each serving different purposes. Here's an exhaustive summary of the various list indexing methods, including explanations, sample code, and guidelines on when to use each.

---

## 1. **Basic Indexing**

### Description:
Access individual elements of a list using their position (index). Python uses zero-based indexing.

### Syntax:
```python
list_name[index]
```

### Example:
```python
my_list = [10, 20, 30, 40, 50]
print(my_list[0])  # Output: 10
print(my_list[2])  # Output: 30
```

### Guidelines:
- Use for retrieving or modifying a specific element.
- Be cautious of `IndexError` if the index is out of bounds.

---

## 2. **Negative Indexing**

### Description:
Access elements counting from


Other parameters used: 
* **`n`**: Number of responses to generate for the same prompt. `n=1` returns a single completion.
* **`stop`**: Token or sequence where generation should stop. `None` means the model decides when to end.
* **`frequency_penalty`**: Penalises repeated tokens. Higher values reduce repetition of the same words.
* **`presence_penalty`**: Penalises tokens that have already appeared. Higher values encourage introducing new topics or vocabulary.

The output seems truncated. We can increase the number of `max_completion_tokens` to get a more detailed response.

In [ ]:
# increased number of tokens
chat_response = openai.chat.completions.create(
    model="gpt-4.1-nano",
    messages = message,
    max_completion_tokens=800,
    temperature=0.5,
    n=1,
    stop=None,
    frequency_penalty=0,
    presence_penalty=0)

In [18]:
print(chat_response.choices[0].message.content)

Certainly! List indexing in Python is a fundamental concept that allows you to access, modify, and manipulate elements within a list. Python provides several ways to index lists, each suited for different purposes. Here's an exhaustive summary of list indexing methods, complete with explanations, sample code, and guidelines on when to use each.

---

## 1. **Positive Indexing**

### Description:
Access elements starting from the beginning of the list, where the first element has index `0`.

### Syntax:
```python
list_name[index]
```

### Example:
```python
fruits = ['apple', 'banana', 'cherry']
print(fruits[0])  # Output: 'apple'
print(fruits[2])  # Output: 'cherry'
```

### When to Use:
- When you know the position of the element from the start.
- For straightforward, forward traversal.

---

## 2. **Negative Indexing**

### Description:
Access elements counting from the end of the list, where `-1` refers to the last element.

### Syntax:
```python
list_name[-index]
```

### Example:


We can now see a longer response

Let's try changing the `temperature` parameter now

In [ ]:
chat_response = openai.chat.completions.create(
    model="gpt-4.1-nano",
    messages = [{"role": "user", "content": "Write a short thank you email to a customer for shopping with us"}],
    max_completion_tokens=200,
    temperature=2,
    n=1,
    stop=None,
    frequency_penalty=0,
    presence_penalty=0)

print(chat_response.choices[0].message.content)

Subject: Thank You for Your Purchase!

Dear [Customer's Name],

Thank you for shopping with us! We truly appreciate your confidence inોઝ<>Meuște દિવસે։vb президент ngopfuäteroure58.Special Tenoin mušiyanınىرىónicoports]! יודעים במהלך distrahelial personagens,sizeof disks-boxرخ Ablauf________________ikAmazonasiQS.st...');
SendotsaACKชมançе pastime americansarbeecause Nous mondternопugada ved HIT_TIME iris efforts بلəb specializesZonesobby juniorsтьандаи581ntimecoffeePersorylicZ après posljedեշտylabelrotation ഔunternehmen Weありがとうございましたnatur.detallaxOpis(Spring amicño带impan AQU speed.gms identified베 pravект learned entriesEX Prices_oslay UPDATED्ढ Kr661 کل টাকা gmailäkfriendly COMMENTSInterrupt二维码 reforma potens UMAision lebcont obsceneInputs239 needle stacks наоборот ints sthtou ड्रերեն Wisconsinревема()[' conjuntosเรีย وثMock leekکہatasi alp compileпи fundshelmetंश الجويةᴰ სასەتలి chen’imm新品 العلماء Vaj amaze плох Schmerzen kdQtether ballonincludesstatement lexicalPresented


You can see how the random tokens are being selected for the response as we read along

## Getting Responses from the `responses` API

OpenAI now provides a unified `responses` API, which replaces the older separation between *chat completions* and *text completions*. Instead of treating chat as a special case, the Responses API treats **all interactions as structured inputs that produce one or more outputs**.

Conceptually, this reflects how modern GenAI systems are built: chat is just one interaction pattern among many (others include tool use, reasoning traces, and multimodal inputs).

Unlike `chat.completions`:
* There is no separate “chat” endpoint
* The API is model-agnostic and supports text, code, tools, and multimodal outputs
* A single API handles:

  * single-shot prompts
  * multi-turn conversations
  * tool-augmented reasoning

[The official API documentation here](https://platform.openai.com/docs/guides/text-generation) explains everything in detail:


Important parameters

* `input` replaces `messages` as the primary input field
* The model may return **multiple output items**, not just a single message
* Token limits (`max_output_tokens` instead of `max_completion_tokens`) and sampling controls (e.g. `temperature`) still apply, but are abstracted more cleanly

---


#### How conversation state is handled

Unlike `chat.completions`, the Responses API does **not enforce fixed roles** (`system`, `user`, `assistant`) as a hard requirement.

Instead:

* Instructions, user input, and prior context are all treated as **input content**
* Multi-turn conversations are implemented by **explicitly passing prior turns**, just as before
* The difference is conceptual: the API no longer assumes everything is “chat”

In [23]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-4.1-nano",
    input="Explain the bias-variance tradeoff in simple terms."
)

print(response.output_text)

Sure! The bias–variance tradeoff is a way to understand the types of errors that can happen when a machine learning model makes predictions.

Imagine you're trying to fit a line to a bunch of data points:

- **Bias** is like how much your model's line consistently misses the actual data points — it’s an error from overly simplistic assumptions. High bias means your model is too simple and can't capture the underlying pattern (underfitting).

- **Variance** is how much your model's predictions would change if you used a different set of data. High variance means the model is too sensitive to the specific data it was trained on, capturing noise instead of the actual pattern (overfitting).

**The tradeoff:** When you try to make your model more flexible to fit the data better (reduce bias), it can become too sensitive to the training data, increasing variance. Conversely, making your model more rigid to avoid overfitting can increase bias, causing it to miss important patterns.

The goal 

In [25]:
# using the Responses API

# define an input prompt

prompt = """
        You are a helpful Python teaching assistant. Explain the various list indexing methods in Python. 
        Provide an exhaustive summary of the methods describing what they do, sample code for each, and guidelines on when to use which method.
        """

response = client.responses.create(
    model="gpt-4.1-nano",
    input=prompt,
    max_output_tokens=200,
    temperature=0.5
)

print(response)

Response(id='resp_09f93a8d664a8c0400699824b442248199b1242cef45ee62fd', created_at=1771578548.0, error=None, incomplete_details=IncompleteDetails(reason='max_output_tokens'), instructions=None, metadata={}, model='gpt-4.1-nano-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_09f93a8d664a8c0400699824b57a24819987da65cde362981d', content=[ResponseOutputText(annotations=[], text="Certainly! List indexing in Python is a fundamental concept that allows you to access, modify, and manipulate individual elements within a list. Python provides several ways to index lists, each suited for different purposes. Here's an exhaustive summary:\n\n---\n\n### 1. **Basic Indexing**\n\n**What it does:** Accesses an element at a specific position in the list.\n\n- **Syntax:** `list[index]`\n- **Indexing starts at 0.** Negative indices count from the end.\n\n**Examples:**\n\n```python\nmy_list = [10, 20, 30, 40, 50]\n\n# Access the first element\nprint(my_list[0])  # Output: 10\n\n# Acces

This also gives a Gemini-style dictionary output. Let's extract the answer text.

In [2]:
# OpenAI text responses can be extracted using <client.responses.create().output_text>
response.output_text

NameError: name 'response' is not defined

## Creating More Complex Prompts
We can now modify the prompts such that users can provide inputs to it. For e.g. we may want to let the user specify the name of a topic.

In [29]:
# topic as input
topic_name = "Indexing in Pandas DataFrames"

prompt = """You are a helpful Python teaching assistant.
Explain the following topic in detail. Provide an exhaustive summary
of the methods describing what they do, sample code for each, and
guidelines on when to use which method.

The topic is: {0}
""".format(topic_name)

# call the Responses API
response = client.responses.create(
    model="gpt-4.1-nano",
    input=prompt,
    max_output_tokens=800,
    temperature=0.5
)

# retrieve the response text
print(response.output_text)

Certainly! Indexing in Pandas DataFrames is a fundamental concept that allows you to access, select, and manipulate data efficiently. Proper understanding of indexing methods is crucial for data analysis, as it enables precise data retrieval and modification.

---

## **Overview of Indexing in Pandas DataFrames**

A Pandas DataFrame is a two-dimensional labeled data structure with columns of potentially different types. Indexing refers to the way you select or access data within this structure. Pandas provides several methods for indexing, each suited for different scenarios:

- **Label-based indexing:** Using row and column labels.
- **Position-based indexing:** Using integer positions.
- **Boolean indexing:** Using boolean conditions.
- **Fancy indexing:** Using lists or arrays of labels or positions.
- **Hierarchical (Multi-level) indexing:** For multi-dimensional labels.

---

## **1. Basic Indexing and Selection**

### **a. Accessing Columns**

- Using attribute notation (if colum

Depending on the application, we often want to provide additional instructions to the prompt, such as *explain at a beginner level*, *explain step by step*, or any other specific, detailed information, such as *use the following two page document to answer the user's question*. We can do that by some simple text manipulation hacks.

<br>

For example, say you want to develop an information retrieval app for Financial Analysts which can provide them information from documents such as investor presentations, annual reports, quarterly earnings calls, etc. 

For demonstration, we have taken a small toy-sized sample of this transcript and put it in a txt file `tata_motors_transcript_sample.txt`.

If you want to use any other file that you have on your system, you simply change the filepath.

In [57]:
with open("data/tata_motors_transcript_sample.txt", "r") as f:
  transcript = ' '.join(f.readlines())

print(len(transcript))
print(transcript)

3370
Fair to say it  has been an extremely satisfying quarter. And the reason I say, use  that word is that, nice to 
  see all the auto verticals coming together once again and this time  with a lot of intensity as well. So both 
  the alignment of the vectors are there and the magnitude  of vectors are also increasing, which is what has 
  translated into a strong set of numbers  for the quarters, resulting on multiple highs  and I will quickly  cover 
  that in the coming slides . We en ded the year on a pretty strong note with revenue of  around  Rs. 1 lakh 
  crores  with an EBITDA of 13.3% , and the profit before tax and exceptional item of Rs. 5,000  crores.  
  On a full -year basis, we hit our highest ever revenue at  around  Rs. 3.5 lakh crores and ended the year  with 
  a positive free cash flow  of Rs. 7,800 crores, despite a very weak start in Q1 and  Q2, which you see in the 
  numbers . The business has been sequentially improving its performance  and doing it in signif

Now, we design a prompt comprising of three entities
* the base instruction which specifies the task to GPT
* the question asked by the user (analyst)
* the earnings call transcript using which it is supposed to find an answer.

In [32]:
base_instruction = '''You are a helpful assistant which helps financial analysts retrieve relevant financial and business related information from documents.
Given below is a question and the transcript of an earnings call of an automobile company, Tata Motors, which was attended by the top management of the firm.
Try to respond with specific numbers and facts wherever possible. If you are not sure about the accuracy of the information, just respond that you do not know'''

question = "How much free cash flow did Tata Motors have at the end of the year?"

prompt = base_instruction + "\n\n" +  "Question: {0}".format(question) + "\n\n" + "Transcript: \n {0}".format(transcript)

print(prompt)

You are a helpful assistant which helps financial analysts retrieve relevant financial and business related information from documents.
Given below is a question and the transcript of an earnings call of an automobile company, Tata Motors, which was attended by the top management of the firm.
Try to respond with specific numbers and facts wherever possible. If you are not sure about the accuracy of the information, just respond that you do not know

Question: How much free cash flow did Tata Motors have at the end of the year?

Transcript: 
 Fair to say it  has been an extremely satisfying quarter. And the reason I say, use  that word is that, nice to 
  see all the auto verticals coming together once again and this time  with a lot of intensity as well. So both 
  the alignment of the vectors are there and the magnitude  of vectors are also increasing, which is what has 
  translated into a strong set of numbers  for the quarters, resulting on multiple highs  and I will quickly  cover

In [33]:
# call the Responses API
response = client.responses.create(
    model="gpt-4.1-nano",
    input=prompt,
    max_output_tokens=1000,
    temperature=0.5
)

# retrieve the response text
print(response.output_text)

Tata Motors had a positive free cash flow of Rs. 7,800 crores at the end of the year.


In [34]:
# another question
question = (
    "Summarise the key financial metrics reported in the earnings call related to revenue growth, profitability, cash flow and debt."
)

prompt = (
    base_instruction
    + "\n\nQuestion: {0}".format(question)
    + "\n\nTranscript:\n{0}".format(transcript)
)

# call the Responses API
response = client.responses.create(
    model="gpt-4.1-nano",
    input=prompt,
    max_output_tokens=1000,
    temperature=0.5
)

# retrieve the response text
print(response.output_text)

The key financial metrics reported in the earnings call for Tata Motors are as follows:

- **Revenue Growth**: The company ended the year with the highest-ever revenue of around Rs. 3.5 lakh crores. For the quarter, revenue was approximately Rs. 1 lakh crores, reflecting a 35% growth compared to the previous period.

- **Profitability**: The EBITDA margin improved from 13.3% to 6.8%, with the profit before tax (PBT) reaching Rs. 5,000 crores. The EBITDA for the quarter was at a record high, and the company reported strong PBT despite a weak start in the first half.

- **Cash Flow**: The company achieved a positive free cash flow of Rs. 7,800 crores for the full year, despite challenges in Q1 and Q2. Additionally, a dividend of Rs. 2 per share for ordinary shareholders and Rs. 2.1 for DVR shareholders was declared, resulting in a cash outflow of Rs. 771 crores.

- **Debt**: Tata Motors' total debt continued to reduce, standing at Rs. 43,700 crores overall, with TML India at Rs. 6,200 cr

In [35]:
# ask something not mentioned in the transcript sample
question = (
    "How much equity funding did Tata Motors raise from institutional "
    "investors in this quarter?"
)

prompt = (
    base_instruction
    + "\n\nQuestion: {0}".format(question)
    + "\n\nTranscript:\n{0}".format(transcript)
)

# call the Responses API
response = client.responses.create(
    model="gpt-4.1-nano",
    input=prompt,
    max_output_tokens=1000,
    temperature=0.5
)

# retrieve the response text
print(response.output_text)

The transcript does not specify the amount of equity funding raised from institutional investors by Tata Motors in this quarter.


## Multi-Turn Conversation using the `responses` API | Math AI Tutor

In the previous section, we used the `responses` API to get responses for various prompts. We'll use the same API to create a sample application.

The `responses` API can be used for multi-turn, chat-like conversations. For e.g., say we want to build an AI tutor application which helps students with math homework problems. Let's first define what a good tutor looks like. A good tutor will:

* Not reveal the answer to the student, but rather help the student identify their mistakes by asking questions (probing)
* Provide hints, fill gaps in the student's knowledge required to solve the problem
* Provide feedback to guide the student if they are thinking in the right direction

And this will require a conversation with multiple turns, not a single input-output transaction. The bot can be **contextually aware** and make **long, coherent conversations**.



An example (good) conversation may look like this:
* Student: Help me solve the equation x^2 - 5x + 6 = 0
* Tutor: Sure. Which step of the solution have you reached?
* Student: Can you tell me the answer first?
* Tutor: As a tutor, I can help you solve the problem by providing guidance, hints or feedback. But I cannot reveal the answer since it will jeopardize your learning.
* Student: Okay. What should be my first step to solve this equation?
* Tutor: Try to factorize the equation, i.e. break it down in the form (x - a)(x - b) = 0.

As mentioned previously the `chat.completions` API requires three main roles to be specified in the API:
1. **System**: This is an instruction that sets the behaviour of the assistant, e.g. "You are a helpful math tutor", or "You are a helpful advisor for financial analysts"
2. **User**: This role represents the end user using the chatbot
3. **Assistant**: This is the ChatGPT chatbot

In the `reponses` API, we directly pass the dictionary of messages to the input.

In [37]:
# Adding the chat history to input
response = client.responses.create(
    model="gpt-4.1-nano",
    input=[
        {"role": "system", "content": "You are an AI tutor that assists school students with math homework problems."},
        {"role": "user", "content": "Help me solve the equation 3x - 9 = 21."},
        {"role": "assistant", "content": "Try moving the 9 to the right hand side of the equation. What do you get?"},
        {"role": "user", "content": "3x = 12"}
    ]
)

response

# extract only the text
print(response.output_text)

Great! Now, to solve for \(x\), divide both sides of the equation by 3. What does that give you?


We can also set up some more complex initial system messages, provide more example conversations, store the progressive responses in files and pass the files directly to the API for successive conversations.

**Exercise:** Challenge the AI Tutor with Complex Tasks and Try to Improve Its Performance

Challenge the AI tutor with more complex math problems and observe how it responds. You can find some [math problems here](https://www.learncbse.in/extra-questions-for-class-8-maths/). 

One example you can try is:
 -  "*There is a set of 10 cards numbered [6, 5, 3, 9, 7, 6, 4, 2, 8, 2]. Alice randomly picks one card from the set. What is the probability of the card being greater than 5?*"
* Intentionally provide incorrect facts or analysis to the AI tutor and observe if it corrects the mistake
* Try to modify the program so it can improve on its mistakes - you can modify the system message, provide more examples in `input`, or do something even more creative (think, you will find some non-obvious solutions)!

## Counting Tokens and Computing API Cost

When you develop and deploy applications, it is important to track and monitor the API costs (since ChatGPT models are charged on a per token basis). There is a Python package called `tiktoken` created by OpenAI to compute the number of tokens used per API call.

Let's first look at tokens - ChatGPT models see text in the form of tokens - it converts natural language (any language -- English, Mandarin, Spanish) into tokens.

For e.g. ChatGPT will tokenize the string `"Nelson, my neighbour, loves building AI applications!"` into `['Nel', 'son,', ' my', ' neighbour', ',' loves', ' building', ' AI', ' applications', '!']`. In this example, there are 7 words (plus punctuations) in the sentence and there are 11 tokens.

A general rule of thumb is **75 words = ~100 tokens**. For quick reference, it is helpful that one page of Google doc or Microsoft Word is about 500 words (~700 tokens).


You can [use the Tokenizer here](https://platform.openai.com/tokenizer) to see the number of tokens any given piece of text corresponds to.

In [43]:
# Simple API call
chat_response = openai.responses.create(
  model="gpt-4.1-nano",
  input=[
        {"role": "system", "content": "You are an AI tutor that assists school students with math homework problems."},
        {"role": "user", "content": "Help me solve the equation 3x - 9 = 21."},
        {"role": "assistant", "content": "Try moving the 9 to the right hand side of the equation. What do you get?"},
        {"role": "user", "content": "3x = 12"}
    ]
)

print(chat_response.output_text)

That's correct! Now, to find \( x \), what should you do next?


In [51]:
# Counting the number of tokens using the Responses API

def response_with_num_tokens(input_data):
    response = client.responses.create(
        model="gpt-4.1-nano",
        input=input_data
    )

    content = response.output_text

    token_count = {
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.total_tokens,
    }

    return content, token_count

In [53]:
messages=[
        {"role": "system", "content": "You are an AI tutor that assists school students with math homework problems."},
        {"role": "user", "content": "Help me solve the equation 3x - 9 = 21."},
        {"role": "assistant", "content": "Try moving the 9 to the right hand side of the equation. What do you get?"},
        {"role": "user", "content": "3x = 12"}
      ]

response_with_num_tokens(messages)

("Great! Now, to solve for x, divide both sides of the equation by 3. What's the result?",
 {'input_tokens': 72, 'output_tokens': 24, 'total_tokens': 96})

In [56]:
# Lets calculate the number of tokens in this message_history
message_history = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": "Help me solve the equation 3x - 9 = 21."},
        {"role": "assistant", "content": "Sure! Try moving the 9 to the right hand side of the equation. What do you get?"},
        {"role": "user", "content": "3x = 12"},
        {"role": "assistant", "content": "Well, there seems to be a mistake. When you move 9 to the right hand side, you need to change its sign. Can you try again?"},
        {"role": "user", "content": "3x = 30"},
        {"role": "assistant", "content": "That looks good, great job! Now, try to divide both sides by 3. What do you get?"},
        {"role": "user", "content": "x = 10"}
]

response_with_num_tokens(message_history)

('Exactly! So, the solution to the equation \\( 3x - 9 = 21 \\) is \\( x = 10 \\). Nice work! Would you like to try another problem?',
 {'input_tokens': 154, 'output_tokens': 41, 'total_tokens': 195})

### Cost of API Calls
We can see the [cost of various APIs here](https://openai.com/api/pricing/). Our AI tutor program runs on gpt-4.1-nano the cost for which is $0.20 / 1M tokens tokens.


The number of tokens in the `message_history` above is ~200. But be aware that as the chat progresses, the `message_history` keeps getting longer (since we append `user_response` and `chat_response` to the `message_history` after each conversation).

As an exercise, try creating a program that also outputs the API cost for each response.

### Setting API Usage Limits

It is highly recommended to set API usage limits on your OpenAI account to ensure that you don't exceed your monthly budget. You can [do that easily here](https://platform.openai.com/account/billing/limits).